# Experiment Bot — Analysis Pipeline
Compute behavioral metrics from bot output and compare to human reference data.

In [ ]:
import hashlib
import json
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
HUMAN_DIR = ROOT / 'data' / 'human'
OUTPUT_DIR = ROOT / 'output'
CACHE_DIR = ROOT / 'cache'

def generate_sub_id(seed_str: str) -> str:
    """Generate a 24-char lowercase hex ID resembling a Prolific subject ID."""
    return hashlib.sha256(seed_str.encode()).hexdigest()[:24]

def load_experiment_data(path):
    """Load experiment data from CSV or JSON."""
    p = Path(path)
    if p.suffix == '.json':
        with open(p) as f:
            return pd.DataFrame(json.load(f))
    return pd.read_csv(p)

def load_cached_config(label):
    """Load cached config and extract Claude's expected parameters."""
    config_path = CACHE_DIR / label / 'config.json'
    if not config_path.exists():
        print(f"  WARNING: No cached config for {label}")
        return None
    with open(config_path) as f:
        return json.load(f)

def display_config_params(config, label):
    """Display Claude's expected behavioral parameters from the cached config."""
    if config is None:
        return
    print(f"=== Claude's Expected Parameters ({label}) ===")
    rd = config.get('response_distributions', {})
    for cond, dist in rd.items():
        p = dist.get('params', {})
        print(f"  {cond}: mu={p.get('mu')}, sigma={p.get('sigma')}, tau={p.get('tau')}")
    perf = config.get('performance', {})
    print(f"  Accuracy targets: {perf.get('accuracy', {})}")
    print(f"  Omission targets: {perf.get('omission_rate', {})}")
    te = config.get('temporal_effects', {})
    enabled = [k for k, v in te.items() if isinstance(v, dict) and v.get('enabled')]
    if enabled:
        print(f"  Temporal effects enabled: {enabled}")
    bsj = config.get('between_subject_jitter', {})
    if bsj.get('rt_mean_sd_ms'):
        print(f"  Between-subject jitter: rt_mean_sd={bsj['rt_mean_sd_ms']}ms")
    print()

def find_latest_run(task_dir_pattern):
    """Find the latest run directory matching a pattern."""
    matches = sorted(OUTPUT_DIR.glob(task_dir_pattern + '/*/experiment_data.*'))
    if not matches:
        return None
    return matches[-1].parent

def summary_table(df, metrics):
    """Mean +/- SD summary for each metric across subjects."""
    rows = []
    for col in metrics:
        if col not in df.columns:
            rows.append({'metric': col, 'mean': float('nan'), 'sd': float('nan'), 'n': 0})
            continue
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        rows.append({'metric': col, 'mean': round(series.mean(), 4), 'sd': round(series.std(ddof=1), 4), 'n': len(series)})
    return pd.DataFrame(rows).set_index('metric')

print(f"ROOT: {ROOT}")
print(f"Output dirs: {sorted([p.name for p in OUTPUT_DIR.iterdir() if p.is_dir()])}")

## Human Reference Data
Load RDoC behavioral battery data, filter to included sessions.

In [ ]:
EXCLUSION_COLS = ['Session-Level Exclusions', 'Task-Level Exclusions', 'Subject-Level Exclusions']

def apply_exclusions(df):
    mask = pd.Series([True] * len(df), index=df.index)
    for col in EXCLUSION_COLS:
        if col in df.columns:
            mask &= df[col].str.strip().eq('Include')
    return df[mask].copy()

ss_human = apply_exclusions(pd.read_csv(HUMAN_DIR / 'stop_signal.csv'))
stroop_human = apply_exclusions(pd.read_csv(HUMAN_DIR / 'stroop.csv'))
stroop_human['stroop_effect'] = (
    pd.to_numeric(stroop_human['incongruent_rt'], errors='coerce')
    - pd.to_numeric(stroop_human['congruent_rt'], errors='coerce')
)

SS_METRICS = ['go_accuracy', 'go_omission_rate', 'go_rt', 'go_rt_all_responses',
              'mean_stop_failure_RT', 'stop_accuracy', 'max_SSD', 'mean_SSD', 'min_SSD', 'final_SSD']
STROOP_METRICS = ['congruent_accuracy', 'congruent_omission_rate', 'congruent_rt',
                  'incongruent_accuracy', 'incongruent_omission_rate', 'incongruent_rt']

print(f"Stop Signal: {len(ss_human)} sessions after exclusions")
print(summary_table(ss_human, SS_METRICS))
print(f"\nStroop: {len(stroop_human)} sessions after exclusions")
print(summary_table(stroop_human, STROOP_METRICS + ['stroop_effect']))

## 1. RDoC Stop Signal (ExpFactory)
Load cached config parameters, compute metrics from experiment output.

In [ ]:
# Load Claude's config
ss_rdoc_config = load_cached_config('expfactory_stop_signal')
display_config_params(ss_rdoc_config, 'expfactory_stop_signal')

# Find latest run
ss_rdoc_dir = find_latest_run('stop_signal_task_(rdoc)')
if ss_rdoc_dir is None:
    print("No RDoC stop signal data found. Run the bot first.")
else:
    data_file = list(ss_rdoc_dir.glob('experiment_data.*'))[0]
    print(f"Loading: {data_file}")
    df = load_experiment_data(data_file)

    test_trials = df[df['trial_id'] == 'test_trial']
    assert len(test_trials) == 180, f'Expected 180 test trials, found {len(test_trials)}'

    go_trials = test_trials[test_trials['condition'] == 'go']
    stop_trials = test_trials[test_trials['condition'] == 'stop']
    assert len(go_trials) == 120, f'Expected 120 go trials, found {len(go_trials)}'
    assert len(stop_trials) == 60, f'Expected 60 stop trials, found {len(stop_trials)}'

    correct_go = go_trials[go_trials['correct_trial'] == 1]
    incorrect_stop = stop_trials[stop_trials['correct_trial'] == 0]

    ss_rdoc_metrics = {
        'go_accuracy': len(correct_go) / len(go_trials),
        'go_omission_rate': go_trials['rt'].isna().mean(),
        'go_rt': correct_go['rt'].mean(),
        'go_rt_all_responses': go_trials['rt'].mean(),
        'mean_stop_failure_RT': incorrect_stop['rt'].mean(),
        'stop_accuracy': (stop_trials['correct_trial'] == 1).mean(),
        'max_SSD': stop_trials['SSD'].max(),
        'mean_SSD': stop_trials['SSD'].mean(),
        'min_SSD': stop_trials['SSD'].min(),
        'final_SSD': stop_trials['SSD'].iloc[-1],
    }

    print("Bot metrics:")
    for k, v in ss_rdoc_metrics.items():
        print(f"  {k}: {v:.4f}")

    # Compare to human
    print("\nBot vs Human:")
    for k in SS_METRICS:
        bot_val = ss_rdoc_metrics.get(k, float('nan'))
        h_mean = pd.to_numeric(ss_human[k], errors='coerce').mean()
        h_sd = pd.to_numeric(ss_human[k], errors='coerce').std()
        within = "YES" if abs(bot_val - h_mean) < h_sd else "NO"
        print(f"  {k:30s}  bot={bot_val:8.3f}  human={h_mean:8.3f} \u00b1{h_sd:7.3f}  within_1SD={within}")

## 2. RDoC Stroop (ExpFactory)
Load cached config parameters, compute metrics from experiment output.

In [ ]:
stroop_rdoc_config = load_cached_config('expfactory_stroop')
display_config_params(stroop_rdoc_config, 'expfactory_stroop')

stroop_rdoc_dir = find_latest_run('stroop_color-word_task_(rdoc)')
if stroop_rdoc_dir is None:
    print("No RDoC Stroop data found. Run the bot first.")
else:
    data_file = list(stroop_rdoc_dir.glob('experiment_data.*'))[0]
    print(f"Loading: {data_file}")
    df = load_experiment_data(data_file)

    test_trials = df[df['trial_id'] == 'test_trial']
    congruent = test_trials[test_trials['condition'] == 'congruent']
    incongruent = test_trials[test_trials['condition'] == 'incongruent']
    assert len(test_trials) == 120, f'Expected 120 test trials, found {len(test_trials)}'
    assert len(congruent) == 60, f'Expected 60 congruent, found {len(congruent)}'
    assert len(incongruent) == 60, f'Expected 60 incongruent, found {len(incongruent)}'

    stroop_rdoc_metrics = {
        'congruent_accuracy': (congruent['correct_trial'] == 1).mean(),
        'congruent_omission_rate': congruent['rt'].isna().mean(),
        'congruent_rt': congruent[congruent['correct_trial'] == 1]['rt'].mean(),
        'incongruent_accuracy': (incongruent['correct_trial'] == 1).mean(),
        'incongruent_omission_rate': incongruent['rt'].isna().mean(),
        'incongruent_rt': incongruent[incongruent['correct_trial'] == 1]['rt'].mean(),
    }

    print("Bot metrics:")
    for k, v in stroop_rdoc_metrics.items():
        print(f"  {k}: {v:.4f}")

    stroop_effect = stroop_rdoc_metrics['incongruent_rt'] - stroop_rdoc_metrics['congruent_rt']
    print(f"  stroop_effect: {stroop_effect:.4f}")

    print("\nBot vs Human:")
    for k in STROOP_METRICS:
        bot_val = stroop_rdoc_metrics.get(k, float('nan'))
        h_mean = pd.to_numeric(stroop_human[k], errors='coerce').mean()
        h_sd = pd.to_numeric(stroop_human[k], errors='coerce').std()
        within = "YES" if abs(bot_val - h_mean) < h_sd else "NO"
        print(f"  {k:30s}  bot={bot_val:8.3f}  human={h_mean:8.3f} \u00b1{h_sd:7.3f}  within_1SD={within}")
    h_se_mean = stroop_human['stroop_effect'].mean()
    h_se_sd = stroop_human['stroop_effect'].std()
    within = "YES" if abs(stroop_effect - h_se_mean) < h_se_sd else "NO"
    print(f"  {'stroop_effect':30s}  bot={stroop_effect:8.3f}  human={h_se_mean:8.3f} \u00b1{h_se_sd:7.3f}  within_1SD={within}")

## 3. STOP-IT Stop Signal
Load cached config parameters, compute metrics from experiment output.

In [ ]:
stopit_config = load_cached_config('stopit_stop_signal')
display_config_params(stopit_config, 'stopit_stop_signal')

stopit_dir = find_latest_run('stop*signal*task*(stop-it)')
if stopit_dir is None:
    print("No STOP-IT data found. Run the bot first.")
else:
    data_file = list(stopit_dir.glob('experiment_data.*'))[0]
    print(f"Loading: {data_file}")
    df = load_experiment_data(data_file)

    go_trials = df[df['signal'] == 'no']
    stop_trials = df[df['signal'] == 'yes']
    assert len(go_trials) == 216, f'Expected 216 go trials, found {len(go_trials)}'
    assert len(stop_trials) == 72, f'Expected 72 stop trials, found {len(stop_trials)}'

    stopit_metrics = {
        'go_accuracy': go_trials['correct'].mean(),
        'go_omission_rate': (go_trials['response'] == 'undefined').mean(),
        'go_rt': go_trials[go_trials['correct'] == True]['rt'].mean(),
        'go_rt_all_responses': go_trials['rt'].mean(),
        'mean_stop_failure_RT': stop_trials[stop_trials['correct'] == False]['rt'].mean(),
        'stop_accuracy': stop_trials['correct'].mean(),
        'max_SSD': stop_trials['SSD'].max(),
        'mean_SSD': stop_trials['SSD'].mean(),
        'min_SSD': stop_trials['SSD'].min(),
        'final_SSD': stop_trials.iloc[-1]['SSD'],
    }

    print("Bot metrics:")
    for k, v in stopit_metrics.items():
        print(f"  {k}: {v:.4f}")

    print("\nBot vs Human (ExpFactory reference):")
    for k in SS_METRICS:
        bot_val = stopit_metrics.get(k, float('nan'))
        h_mean = pd.to_numeric(ss_human[k], errors='coerce').mean()
        h_sd = pd.to_numeric(ss_human[k], errors='coerce').std()
        within = "YES" if abs(bot_val - h_mean) < h_sd else "NO"
        print(f"  {k:30s}  bot={bot_val:8.3f}  human={h_mean:8.3f} \u00b1{h_sd:7.3f}  within_1SD={within}")

## 4. Cognition.run Stroop
Load cached config parameters, compute metrics from experiment output.

In [ ]:
cogrun_config = load_cached_config('cognitionrun_stroop')
display_config_params(cogrun_config, 'cognitionrun_stroop')

cogrun_dir = find_latest_run('stroop_color-word_task')
# Exclude rdoc variant
if cogrun_dir and '(rdoc)' in str(cogrun_dir):
    dirs = sorted(OUTPUT_DIR.glob('stroop_color-word_task/*/experiment_data.*'))
    cogrun_dir = dirs[-1].parent if dirs else None

if cogrun_dir is None:
    print("No Cognition.run Stroop data found. Run the bot first.")
else:
    data_file = list(cogrun_dir.glob('experiment_data.*'))[0]
    print(f"Loading: {data_file}")
    df = load_experiment_data(data_file)

    test_trials = df[df['text'].notna()].copy()
    assert len(test_trials) == 15, f'Expected 15 test trials, found {len(test_trials)}'

    test_trials['trial_condition'] = test_trials.apply(
        lambda row: 'congruent' if str(row['text']).lower() == str(row['colour']).lower() else 'incongruent', axis=1
    )
    test_trials['correct_trial'] = test_trials.apply(
        lambda row: 1 if str(row['response']).lower() == str(row['colour'])[0].lower() else 0, axis=1
    )

    congruent = test_trials[test_trials['trial_condition'] == 'congruent']
    incongruent = test_trials[test_trials['trial_condition'] == 'incongruent']

    cogrun_metrics = {
        'congruent_accuracy': congruent['correct_trial'].mean() if len(congruent) > 0 else float('nan'),
        'congruent_omission_rate': congruent['rt'].isna().mean() if len(congruent) > 0 else float('nan'),
        'congruent_rt': congruent[congruent['correct_trial'] == 1]['rt'].mean() if len(congruent) > 0 else float('nan'),
        'incongruent_accuracy': incongruent['correct_trial'].mean() if len(incongruent) > 0 else float('nan'),
        'incongruent_omission_rate': incongruent['rt'].isna().mean() if len(incongruent) > 0 else float('nan'),
        'incongruent_rt': incongruent[incongruent['correct_trial'] == 1]['rt'].mean() if len(incongruent) > 0 else float('nan'),
    }

    print(f"Bot metrics ({len(congruent)} congruent, {len(incongruent)} incongruent):")
    for k, v in cogrun_metrics.items():
        print(f"  {k}: {v:.4f}" if not np.isnan(v) else f"  {k}: N/A")

    if not np.isnan(cogrun_metrics['congruent_rt']) and not np.isnan(cogrun_metrics['incongruent_rt']):
        stroop_effect = cogrun_metrics['incongruent_rt'] - cogrun_metrics['congruent_rt']
        print(f"  stroop_effect: {stroop_effect:.4f}")

    print("\nBot vs Human (ExpFactory reference):")
    for k in STROOP_METRICS:
        bot_val = cogrun_metrics.get(k, float('nan'))
        h_mean = pd.to_numeric(stroop_human[k], errors='coerce').mean()
        h_sd = pd.to_numeric(stroop_human[k], errors='coerce').std()
        if np.isnan(bot_val):
            print(f"  {k:30s}  bot=     N/A  human={h_mean:8.3f} \u00b1{h_sd:7.3f}")
        else:
            within = "YES" if abs(bot_val - h_mean) < h_sd else "NO"
            print(f"  {k:30s}  bot={bot_val:8.3f}  human={h_mean:8.3f} \u00b1{h_sd:7.3f}  within_1SD={within}")

## Export Bot Metrics as CSV
Save per-run metrics in the same format as the human reference data, with generated subject IDs.

In [ ]:
OUTPUT_CSV_DIR = ROOT / 'data' / 'bot'
OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)

now_str = datetime.now().strftime('%m/%d/%Y')

# --- Stop Signal CSV ---
ss_rows = []

if ss_rdoc_dir is not None:
    run_ts = ss_rdoc_dir.name
    ss_rows.append({
        'sub_id': generate_sub_id(f"ss_rdoc_{run_ts}"),
        'date_time': now_str,
        'session': 1,
        'platform': 'expfactory',
        **ss_rdoc_metrics,
    })

if stopit_dir is not None:
    run_ts = stopit_dir.name
    ss_rows.append({
        'sub_id': generate_sub_id(f"stopit_{run_ts}"),
        'date_time': now_str,
        'session': 1,
        'platform': 'stopit',
        **stopit_metrics,
    })

if ss_rows:
    ss_bot_df = pd.DataFrame(ss_rows)
    col_order = ['sub_id', 'date_time', 'session', 'platform'] + SS_METRICS
    ss_bot_df = ss_bot_df[[c for c in col_order if c in ss_bot_df.columns]]
    ss_bot_df.to_csv(OUTPUT_CSV_DIR / 'stop_signal.csv', index=False)
    print(f"Saved {len(ss_bot_df)} stop signal bot rows to data/bot/stop_signal.csv")
    print(ss_bot_df.to_string(index=False))
else:
    print("No stop signal bot data to export.")

# --- Stroop CSV ---
stroop_rows = []

if stroop_rdoc_dir is not None:
    run_ts = stroop_rdoc_dir.name
    stroop_rows.append({
        'sub_id': generate_sub_id(f"stroop_rdoc_{run_ts}"),
        'date_time': now_str,
        'session': 1,
        'platform': 'expfactory',
        **stroop_rdoc_metrics,
    })

if cogrun_dir is not None:
    run_ts = cogrun_dir.name
    stroop_rows.append({
        'sub_id': generate_sub_id(f"cogrun_{run_ts}"),
        'date_time': now_str,
        'session': 1,
        'platform': 'cognitionrun',
        **cogrun_metrics,
    })

if stroop_rows:
    stroop_bot_df = pd.DataFrame(stroop_rows)
    col_order = ['sub_id', 'date_time', 'session', 'platform'] + STROOP_METRICS
    stroop_bot_df = stroop_bot_df[[c for c in col_order if c in stroop_bot_df.columns]]
    stroop_bot_df.to_csv(OUTPUT_CSV_DIR / 'stroop.csv', index=False)
    print(f"\nSaved {len(stroop_bot_df)} Stroop bot rows to data/bot/stroop.csv")
    print(stroop_bot_df.to_string(index=False))
else:
    print("No Stroop bot data to export.")

## Summary
All metrics computed and exported. Bot CSVs saved to `data/bot/` in the same format as human reference data.